In [1]:
import numpy as np
import pandas as pd
import gymnasium as gym

# Adjust these imports to your project layout
import env_config
from battery_env import BatteryEnv 

# Instantiate env
env = BatteryEnv(
    env_config,
    publish_hour=13,
    episode_days=7,              
    randomize_init_soc=False,  
    seed=123,
)

obs, info = env.reset(seed=123)
print("obs shape:", np.array(obs).shape)
print("info:", info)
print("action_space:", env.action_space)
print("observation_space:", env.observation_space)


obs shape: (54,)
info: {'start_day': '2021-01-12T00:00:00.000000000', 'publish_hour': 13, 'episode_days': 7, 'init_soc': 0.5}
action_space: MultiDiscrete([11 11])
observation_space: Box([ 0.0000e+00 -3.0000e+02  0.0000e+00  1.0000e+00 -1.8635e-01  0.0000e+00
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02
 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02 -3.0000e+02], [1.0000e+00 2.5000e+03 1.0000e+03 4.8000e+01 1.8635e-01 1.0000e+00
 2.5000e+03 2.5000e+03 2.5000e+03 2.5000e+03 2.5000e+03 2.5000e+03
 2.5000e+03 2.5000e+0

## 1.1. Quick Environment Structure Check

In [3]:
import numpy as np
import pandas as pd
import gymnasium as gym

# Adjust these imports to your project layout
import env_config
from battery_env import BatteryEnv 

# Instantiate env
env = BatteryEnv(
    env_config,
    publish_hour=13,
    episode_days=7,              
    randomize_init_soc=False,  
    seed=123,
)

obs, info = env.reset(seed=123)

action = env.action_space.sample()
obs, r, term, trunc, info = env.step(action)

print("Obs shape: ", obs.shape)
print("Obs length: ", len(obs))
print("Expected length: ", 54)
assert len(obs) == 54, "Obs length mismatch; exepcted 54 = 6  main + 48 curve"

print("Obs space low/high lengths:", len(env.observation_space.low), len(env.observation_space.high))
assert len(env.observation_space.low) == 54
assert len(env.observation_space.high) == 54

print("valid days:", len(env.valid_days))
print("first valid days: ", env.valid_days[0], "last:", env.valid_days[-1])

assert (obs[6:] != 0).any() == info["da_available"]


Obs shape:  (54,)
Obs length:  54
Expected length:  54
Obs space low/high lengths: 54 54
valid days: 728
first valid days:  2021-01-01T00:00:00.000000000 last: 2022-12-31T00:00:00.000000000


## 1.2. Pick one Day and Print its 48 slots (table)

In [4]:
#pick the current day the env started on
day = env.current_day if hasattr(env, "current_day") else env.current_delivery_day
idxs = env.day_indices[day]

df_day = env.df.loc[idxs, ["trade_ts", "delivery_date", "tau", "price_gbp_mwh", "carbon_gco2_kwh"]].copy()
df_day = df_day.sort_values("trade_ts").reset_index(drop=True)

display(df_day.head(10))
display(df_day.tail(10))

# Hard assertions
assert len(df_day) == 48, f"Day has {len(df_day)} rows, exepcted 48"
assert set(df_day["tau"].tolist()) == set(range(1,49)), "tau is not exactly 1,48"

deltas = df_day["trade_ts"].diff().dropna()
assert (deltas == pd.Timedelta(minutes=30)).all(), "timestamps  are not exaclty 30 minutes "

print ("Day check passed:", day)

,trade_ts,delivery_date,tau,price_gbp_mwh,carbon_gco2_kwh
0,2021-01-12 00:00:00,2021-01-13,1,59.5,149.0
1,2021-01-12 00:30:00,2021-01-13,2,72.5,149.0
2,2021-01-12 01:00:00,2021-01-13,3,65.4,149.0
3,2021-01-12 01:30:00,2021-01-13,4,65.0,140.0
4,2021-01-12 02:00:00,2021-01-13,5,65.1,131.0
5,2021-01-12 02:30:00,2021-01-13,6,62.3,132.0
6,2021-01-12 03:00:00,2021-01-13,7,65.4,141.0
7,2021-01-12 03:30:00,2021-01-13,8,62.0,138.0
8,2021-01-12 04:00:00,2021-01-13,9,64.0,137.0
9,2021-01-12 04:30:00,2021-01-13,10,60.0,143.0


,trade_ts,delivery_date,tau,price_gbp_mwh,carbon_gco2_kwh
38,2021-01-12 19:00:00,2021-01-13,39,199.0,284.0
39,2021-01-12 19:30:00,2021-01-13,40,96.6,286.0
40,2021-01-12 20:00:00,2021-01-13,41,102.0,292.0
41,2021-01-12 20:30:00,2021-01-13,42,74.8,293.0
42,2021-01-12 21:00:00,2021-01-13,43,80.0,295.0
43,2021-01-12 21:30:00,2021-01-13,44,66.5,296.0
44,2021-01-12 22:00:00,2021-01-13,45,65.4,296.0
45,2021-01-12 22:30:00,2021-01-13,46,60.0,287.0
46,2021-01-12 23:00:00,2021-01-13,47,65.1,279.0
47,2021-01-12 23:30:00,2021-01-13,48,64.8,277.0


Day check passed: 2021-01-12T00:00:00.000000000


## 1.3. Validate all valid days match the rules

In [5]:
bad = []
for d in env.valid_days:
    idxs = env.day_indices[d]
    df_d = env.df.loc[idxs, ["trade_ts", "tau"]].sort_values("trade_ts")
    if len(df_d) != 48:
        bad.append((d, "len", len(df_d)))
        continue
    if set(df_d["tau"].tolist()) != set(range(1,49)):
        bad.append(d, "tau_set", sorted(set(df_d["tau"].tolist())[:10]))
        continue
    deltas = df_d["trade_ts"].diff().dropna()
    if not(deltas == pd.Timedelta(minutes=30)).all():
        bad.append(d, "cadence", deltas.value_counts().head(3).to_dict())
        continue

print('Bad days count:', len(bad))
if bad:
    print("First 5 bad:",bad[:5])
assert len(bad) == 0, "Some valid days violate 48/tau/cadence rules."
print("All valids_days passed integrity checks")



Bad days count: 0
All valids_days passed integrity checks


In [6]:
if max(env.tau) == 48:
    print("Tau = 48")
else:
    print("Tay XX<=48")

if min(env.tau) == 1:
    print("Tau = 1")
else:
    print("Tau XXX>0")

if min(env.ci) >= 0:
    print("CI Min > 0")
else:
    print("CI Min XXX>0")

if max(env.ci) <= 1000:
    print("CI Max < 1000")
else:
    print("CI Max XXX<1000")

Tau = 48
Tau = 1
CI Min > 0
CI Max < 1000


In [2]:
import numpy as np
import pandas as pd
import env_config
from battery_env import BatteryEnv

def test_minus1_means_no_plan(env, N=48, seed=123):
    obs, info = env.reset(seed=seed)

    # Force: no DA commitments for the whole day
    env.today_plan[:] = -1
    env.tomorrow_plan[:] = -1

    rows = []
    fails = 0

    for k in range(N):
        a = env.action_space.sample()
        agent_dispatch = int(a[0])

        obs, r, term, trunc, info = env.step(a)

        # You should add this to info in env.step() for clean testing:
        # info["dispatch_idx_exec"] = dispatch_idx_eff
        exec_dispatch = info.get("dispatch_idx_exec", None)

        # If you didn't add it, we can infer from P_req_MW (less robust if protection clips),
        # so strongly recommend adding dispatch_idx_exec to info.
        if exec_dispatch is None:
            raise RuntimeError("Add info['dispatch_idx_exec'] in env.step() to run this test cleanly.")

        ok = (exec_dispatch == agent_dispatch)
        fails += (not ok)

        rows.append({
            "k": k,
            "tau": info["tau"],
            "agent_dispatch": agent_dispatch,
            "exec_dispatch": exec_dispatch,
            "ok": ok,
            "P_req_MW": info["P_req_MW"],
            "P_app_MW": info["P_applied_MW"],
        })

        if term or trunc:
            break

    df = pd.DataFrame(rows)
    print("fails:", fails, "out of", len(df))
    display(df[df["ok"] == False].head(10))
    return df

env = BatteryEnv(env_config)
df = test_minus1_means_no_plan(env)

fails: 0 out of 48


,k,tau,agent_dispatch,exec_dispatch,ok,P_req_MW,P_app_MW


In [3]:
def get_idle_index(env):
    return int(np.argmin(np.abs(env.power_levels)))

env = BatteryEnv(env_config)
idle_idx = get_idle_index(env)
print("idle_idx:", idle_idx, "power:", env.power_levels[idle_idx])

idle_idx: 5 power: 0.0


In [5]:
import numpy as np
import pandas as pd
import env_config
from battery_env import BatteryEnv

def test_plan_idle_overrides_agent(env, N=20, seed=123):
    obs, info = env.reset(seed=seed)

    idle_idx = int(np.argmin(np.abs(env.power_levels)))

    # Force: every slot has a planned idle commitment
    env.today_plan[:] = idle_idx

    rows = []
    fails = 0

    for k in range(N):
        a = env.action_space.sample()
        agent_dispatch = int(a[0])

        obs, r, term, trunc, info = env.step(a)
        exec_dispatch = info.get("dispatch_idx_exec", None)

        if exec_dispatch is None:
            raise RuntimeError("Add info['dispatch_idx_exec'] in env.step()")

        ok = (exec_dispatch == idle_idx)  # should ALWAYS be idle
        fails += (not ok)

        rows.append({
            "k": k,
            "tau": info["tau"],
            "agent_dispatch": agent_dispatch,
            "exec_dispatch": exec_dispatch,
            "idle_idx": idle_idx,
            "ok": ok,
            "P_req_MW": info["P_req_MW"],
            "P_app_MW": info["P_applied_MW"],
        })

        if term or trunc:
            break

    df = pd.DataFrame(rows)
    print("fails:", fails, "out of", len(df))
    display(df[df["ok"] == False].head(10))
    return df
print(df)

env = BatteryEnv(env_config)
df = test_plan_idle_overrides_agent(env)

     k  tau  agent_dispatch  exec_dispatch  idle_idx    ok  P_req_MW  P_app_MW
0    0    1               2              5         5  True       0.0       0.0
1    1    2               5              5         5  True       0.0       0.0
2    2    3               6              5         5  True       0.0       0.0
3    3    4               3              5         5  True       0.0       0.0
4    4    5               1              5         5  True       0.0       0.0
5    5    6              10              5         5  True       0.0       0.0
6    6    7               6              5         5  True       0.0       0.0
7    7    8               6              5         5  True       0.0       0.0
8    8    9               4              5         5  True       0.0       0.0
9    9   10               2              5         5  True       0.0       0.0
10  10   11               1              5         5  True       0.0       0.0
11  11   12               6              5         5

,k,tau,agent_dispatch,exec_dispatch,idle_idx,ok,P_req_MW,P_app_MW


# 2. Timeline Correctness

## 2.1. Is the step through time done correctly

This test verifies that one environment step corresponds to exactly one 30-minute
interval in the dataset. We check that timestamps advance correctly, tau increments
from 1 to 48 without skips, and the environment remains within the same delivery day
until a full day (48 steps) is completed. This ensures that the temporal structure of
the environment is consistent with the physical market timeline.

In [7]:
import pandas as pd
import numpy as np


def run_time_step_sanity(env, n_steps=60, seed=123):
    obs, info = env.reset(seed=seed)
    print("Reset info:", info)
    print("Start trade day (env.current_day):", env.current_day, "| slot0:", env.slot0, "| days_done:", env.days_done)
    print()

    rows = []
    for k in range(n_steps):
        obs_before = obs.copy()                 # <- state s_t
        action = env.action_space.sample()
        obs, r, term, trunc, inf = env.step(action)  # obs is now s_{t+1}

        rows.append({
            "k": k,
            "trade_ts": inf["trade_ts"],
            "trade_day": pd.to_datetime(inf["trade_ts"]).floor("D"),
            "tau": inf["tau"],
            "da_available": inf["da_available"],
            "tomorrow_curve_nonzero": bool(np.any(obs_before[6:] != 0.0)),
        })

        if term or trunc:
            break

    df_print = pd.DataFrame(rows)

    # 1) 30-min cadence check on trade_ts
    ts = pd.to_datetime(df_print["trade_ts"])
    df_print["dt_min"] = ts.diff().dt.total_seconds() / 60.0

    # 2) expected DA gate time for the first trade day we are in
    # DA should become available at trade_day 13:00 (or your env.publish_hour)
    trade_day0 = pd.to_datetime(df_print["trade_day"].iloc[0]).floor("D")
    publish_ts = trade_day0 + pd.Timedelta(hours=env.publish_hour)
    df_print["publish_ts_expected"] = str(publish_ts)

    return df_print

df_t = run_time_step_sanity(env, n_steps=60, seed=123)
df_t.head(35)

Reset info: {'start_day': '2021-01-12T00:00:00.000000000', 'publish_hour': 13, 'episode_days': 7, 'init_soc': 0.5}
Start trade day (env.current_day): 2021-01-12T00:00:00.000000000 | slot0: 0 | days_done: 0



,k,trade_ts,trade_day,tau,da_available,tomorrow_curve_nonzero,dt_min,publish_ts_expected
0,0,2021-01-12 00:00:00,2021-01-12,1,False,False,NaN,2021-01-12 13:00:00
1,1,2021-01-12 00:30:00,2021-01-12,2,False,False,30.0,2021-01-12 13:00:00
2,2,2021-01-12 01:00:00,2021-01-12,3,False,False,30.0,2021-01-12 13:00:00
3,3,2021-01-12 01:30:00,2021-01-12,4,False,False,30.0,2021-01-12 13:00:00
4,4,2021-01-12 02:00:00,2021-01-12,5,False,False,30.0,2021-01-12 13:00:00
5,5,2021-01-12 02:30:00,2021-01-12,6,False,False,30.0,2021-01-12 13:00:00
6,6,2021-01-12 03:00:00,2021-01-12,7,False,False,30.0,2021-01-12 13:00:00
7,7,2021-01-12 03:30:00,2021-01-12,8,False,False,30.0,2021-01-12 13:00:00
8,8,2021-01-12 04:00:00,2021-01-12,9,False,False,30.0,2021-01-12 13:00:00
9,9,2021-01-12 04:30:00,2021-01-12,10,False,False,30.0,2021-01-12 13:00:00


## 2.2. SoC Physics Sanity 

In this test, I run the environment for a large number of steps using randomly sampled actions to stress-test the battery dynamics. At each timestep, I verify that the state of charge (SoC) remains within the physical operating limits defined by SoC_min and SoC_max, allowing for a small numerical tolerance.

This check ensures that:
	•	the SoC protection logic correctly prevents over-charging and over-discharging,
	•	the current and power clamping in the battery model are consistent with the Coulomb-counting update,
	•	no numerical instability (e.g. NaNs or drift outside bounds) appears over long simulations.

If a violation is detected, the test logs the timestep, action taken, requested and applied power, reward, and timestamp, and stops immediately to simplify debugging.

In [8]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)

eps = 1e-6
N_steps = 5000

obs, info = env.reset(seed=123)
rows = []
fails = []

for t in range(N_steps):
    soc_before = float(env.soc)

    a = env.action_space.sample()          # array([dispatch_idx, plan_idx])
    dispatch_idx = int(a[0])
    plan_idx = int(a[1])

    obs, reward, terminated, truncated, info = env.step(a)

    soc_after = float(env.soc)
    delta = soc_after - soc_before

    P_req = float(env.power_levels[dispatch_idx])
    P_app = float(info.get("P_applied_MW", np.nan))

    # --- sign / direction sanity ---
    ok = True

    # Charge (negative power) => SoC should increase (or stay flat if blocked at max)
    if P_app < -eps and not (soc_after >= soc_before - 1e-8):
        ok = False

    # Discharge (positive power) => SoC should decrease (or stay flat if blocked at min)
    if P_app > eps and not (soc_after <= soc_before + 1e-8):
        ok = False

    # Near-zero power => SoC should barely move
    if abs(P_app) <= eps and not (abs(delta) <= 1e-6):
        ok = False

    row = {
        "t": t,
        "dispatch_idx": dispatch_idx,
        "plan_idx": plan_idx,
        "P_req_MW": P_req,
        "P_app_MW": P_app,
        "soc_before": soc_before,
        "soc_after": soc_after,
        "delta_soc": delta,
        "ok_sign": ok,
        "reward": float(reward),
        "trade_ts": info.get("trade_ts", None),
        "tau": info.get("tau", None),
        "da_available": info.get("da_available", None),
    }
    rows.append(row)

    if not ok:
        fails.append(row)
        break

    if terminated or truncated:
        break

df = pd.DataFrame(rows)
print("ran steps:", len(df))
print("num fails:", len(fails))
if fails:
    display(pd.DataFrame(fails))
else:
    display(df.head(20))

ran steps: 1440
num fails: 0


,t,dispatch_idx,plan_idx,P_req_MW,P_app_MW,soc_before,soc_after,delta_soc,ok_sign,reward,trade_ts,tau,da_available
0,0,7,4,0.07454,0.074540,0.572941,0.475261,-0.097679,True,1613.790928,2022-02-19 00:00:00,1,False
1,1,4,9,-0.03727,-0.037270,0.475261,0.523251,0.047990,True,-823.890583,2022-02-19 00:30:00,2,False
2,2,10,5,0.18635,0.186350,0.523251,0.276058,-0.247193,True,4541.349500,2022-02-19 01:00:00,3,False
3,3,3,3,-0.07454,-0.074540,0.276058,0.372176,0.096117,True,-1780.760521,2022-02-19 01:30:00,4,False
4,4,4,5,-0.03727,-0.037270,0.372176,0.420188,0.048012,True,-907.479736,2022-02-19 02:00:00,5,False
5,5,1,4,-0.14908,-0.149080,0.420188,0.610358,0.190170,True,-3762.033633,2022-02-19 02:30:00,6,False
6,6,4,2,-0.03727,-0.037270,0.610358,0.658082,0.047724,True,-957.622791,2022-02-19 03:00:00,7,False
7,7,7,8,0.07454,0.074540,0.658082,0.561287,-0.096795,True,1913.926225,2022-02-19 03:30:00,8,False
8,8,5,6,0.00000,0.000000,0.561287,0.561287,0.000000,True,0.000000,2022-02-19 04:00:00,9,False
9,9,4,6,-0.03727,-0.037270,0.561287,0.609194,0.047907,True,-956.907207,2022-02-19 04:30:00,10,False


### 2.2.1. Charging Battery

In [9]:
charge_idx = 0                 # most negative power
discharge_idx = env.n_power_levels - 1  # most positive power

obs, info = env.reset(seed=123)

soc_before = env.soc

action = np.array([charge_idx, 0])  # dispatch charge, ignore planning
obs, r, term, trunc, info = env.step(action)

soc_after = env.soc

print("SoC before:", soc_before)
print("SoC after :", soc_after)
print("ΔSoC      :", soc_after - soc_before)
print("Applied MW:", info["P_applied_MW"])

SoC before: 0.5729407452992573
SoC after : 0.8092611446495056
ΔSoC      : 0.2363203993502483
Applied MW: -0.18635


### 2.2.2. Discharging battery

In [10]:
charge_idx = 0                 # most negative power
discharge_idx = env.n_power_levels - 1  # most positive power

obs, info = env.reset(seed=123)

soc_before = env.soc

action = np.array([discharge_idx, 0])  # dispatch discharge, ignore planning
obs, r, term, trunc, info = env.step(action)

soc_after = env.soc

print("SoC before:", soc_before)
print("SoC after :", soc_after)
print("ΔSoC      :", soc_after - soc_before)
print("Applied MW:", info["P_applied_MW"])

SoC before: 0.5729407452992573
SoC after : 0.3262280117363327
ΔSoC      : -0.2467127335629246
Applied MW: 0.18635


# 3. Sign Consistency: Power and SoC

In [11]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)
eps = 1e-9

def idx_for_power(env, target_MW: float) -> int:
    return int(np.argmin(np.abs(env.power_levels - target_MW)))

dispatch_charge = idx_for_power(env, -env.P_max_MW)
dispatch_idle   = idx_for_power(env, 0.0)
dispatch_dis    = idx_for_power(env,  env.P_max_MW)

# plan_idx can be anything for this test; keep it simple
plan_idx_const = dispatch_idle

# Build a sequence of 2D actions: [dispatch_idx, plan_idx]
plan = (
    [np.array([dispatch_charge, plan_idx_const], dtype=np.int64)] * 5
    + [np.array([dispatch_idle,   plan_idx_const], dtype=np.int64)] * 3
    + [np.array([dispatch_dis,    plan_idx_const], dtype=np.int64)] * 5
)

obs, info = env.reset()

rows = []
fails = []

for t, a in enumerate(plan):
    soc_before = float(env.soc)

    obs, reward, terminated, truncated, info = env.step(a)

    soc_after = float(env.soc)
    P_app = float(info["P_applied_MW"])
    delta = soc_after - soc_before

    ok = True
    # Negative power => charge => SoC should go up (or stay ~same if blocked at max)
    if P_app < -eps and soc_after < soc_before - 1e-8:
        ok = False
    # Positive power => discharge => SoC should go down (or stay ~same if blocked at min)
    if P_app > eps and soc_after > soc_before + 1e-8:
        ok = False
    # ~0 power => SoC should barely move
    if abs(P_app) <= eps and abs(delta) > 1e-6:
        ok = False

    dispatch_idx = int(a[0])
    plan_idx = int(a[1])

    row = {
        "t": t,
        "dispatch_idx": dispatch_idx,
        "plan_idx": plan_idx,
        "P_req_MW": float(env.power_levels[dispatch_idx]),
        "P_app_MW": P_app,
        "soc_before": soc_before,
        "soc_after": soc_after,
        "delta_soc": delta,
        "ok_sign": ok,
        "reward": float(reward),
        "trade_ts": info.get("trade_ts", None),
        "tau": info.get("tau", None),
        "da_available": info.get("da_available", None),
    }
    rows.append(row)
    if not ok:
        fails.append(row)

    if terminated or truncated:
        break

df = pd.DataFrame(rows)
display(df)

print("num sign fails:", len(fails))
if fails:
    display(pd.DataFrame(fails))

,t,dispatch_idx,plan_idx,P_req_MW,P_app_MW,soc_before,soc_after,delta_soc,ok_sign,reward,trade_ts,tau,da_available
0,0,0,5,-0.18635,-1.642026e-01,0.693274,0.900000,2.067259e-01,True,-1.390139e+04,2022-10-14 00:00:00,1,False
1,1,0,5,-0.18635,-6.091568e-16,0.900000,0.900000,1.221245e-15,True,-5.212098e-11,2022-10-14 00:30:00,2,False
2,2,0,5,-0.18635,0.000000e+00,0.900000,0.900000,0.000000e+00,True,0.000000e+00,2022-10-14 01:00:00,3,False
3,3,0,5,-0.18635,0.000000e+00,0.900000,0.900000,0.000000e+00,True,0.000000e+00,2022-10-14 01:30:00,4,False
4,4,0,5,-0.18635,0.000000e+00,0.900000,0.900000,0.000000e+00,True,0.000000e+00,2022-10-14 02:00:00,5,False
5,5,5,5,0.00000,0.000000e+00,0.900000,0.900000,0.000000e+00,True,0.000000e+00,2022-10-14 02:30:00,6,False
6,6,5,5,0.00000,0.000000e+00,0.900000,0.900000,0.000000e+00,True,0.000000e+00,2022-10-14 03:00:00,7,False
7,7,5,5,0.00000,0.000000e+00,0.900000,0.900000,0.000000e+00,True,0.000000e+00,2022-10-14 03:30:00,8,False
8,8,10,5,0.18635,1.863500e-01,0.900000,0.655846,-2.441536e-01,True,1.745352e+04,2022-10-14 04:00:00,9,False
9,9,10,5,0.18635,1.863500e-01,0.655846,0.411407,-2.444394e-01,True,1.778732e+04,2022-10-14 04:30:00,10,False


num sign fails: 0


# 4. SOC Protection Function 

In [12]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)

eps = 1e-6

# --- SoC grid to test edge + interior ---
soc_test = np.array(
    [
        env.SoC_min,
        env.SoC_min + 1e-3,
        0.2,
        0.5,
        0.9,
        env.SoC_max - 1e-3,
        env.SoC_max,
    ],
    dtype=float
)

# --- Requested power grid ---
power_requested = np.array(
    [
        -env.P_max_MW,
        -0.5 * env.P_max_MW,
        0.0,
        0.5 * env.P_max_MW,
        env.P_max_MW,
    ],
    dtype=float
)

rows = []

for soc in soc_test:
    for P_req in power_requested:

        # --- Call protection ---
        P_app, I_app, V_oc = env._apply_soc_protection(P_req, soc)

        # --- Hard power bounds ---
        ok_power_bounds = abs(P_app) <= env.P_max_MW + eps

        # --- Detect SoC boundary ---
        at_max = soc >= env.SoC_max - 1e-12
        at_min = soc <= env.SoC_min + 1e-12

        # --- Correct blocking behaviour ---
        blocked_charge_at_max = (
            at_max and P_req < 0 and abs(P_app) < eps and abs(I_app) < eps
        )

        blocked_discharge_at_min = (
            at_min and P_req > 0 and abs(P_app) < eps and abs(I_app) < eps
        )

        # --- Compute implied SoC change ---
        if I_app < 0:   # charging
            delta_soc = -(I_app * env.dt_seconds / env.Q_pack_C) * env.eta_ch
        elif I_app > 0: # discharging
            delta_soc = -(I_app * env.dt_seconds / env.Q_pack_C) / env.eta_dis
        else:
            delta_soc = 0.0

        soc_next = soc + delta_soc
        ok_soc_bounds = env.SoC_min - eps <= soc_next <= env.SoC_max + eps

        # --- Power/current consistency (ECM check) ---
        P_recon_W = I_app * (V_oc - I_app * env.R_sys)
        P_recon_MW = P_recon_W / 1e6
        ok_power_consistency = abs(P_recon_MW - P_app) <= 1e-4

        changed = abs(P_app - P_req) > 1e-9

        rows.append({
            "soc": soc,
            "P_req_MW": P_req,
            "P_app_MW": P_app,
            "deltaP": P_app - P_req,
            "changed": changed,
            "I_app_A": I_app,
            "V_oc_V": V_oc,
            "P_reconstructed_MW": P_recon_MW,
            "soc_next": soc_next,
            "ok_power_bounds": ok_power_bounds,
            "ok_soc_bounds": ok_soc_bounds,
            "ok_power_consistency": ok_power_consistency,
            "blocked_charge_at_max": blocked_charge_at_max,
            "blocked_discharge_at_min": blocked_discharge_at_min,
        })

df = pd.DataFrame(rows).sort_values(["soc", "P_req_MW"]).reset_index(drop=True)

# ---- Display full table ----
print("FULL SoC PROTECTION TEST RESULTS")
display(df)

# ---- Failures ----
mask_ok = (
    df["ok_power_bounds"]
    & df["ok_soc_bounds"]
    & df["ok_power_consistency"]
)

fails = df.loc[~mask_ok].reset_index(drop=True)

print(f"num tests: {len(df)} | num fails: {len(fails)}")
display(fails)

# Optional export
df.to_csv("soc_protection_all.csv", index=False)
fails.to_csv("soc_protection_fails.csv", index=False)

FULL SoC PROTECTION TEST RESULTS


,soc,P_req_MW,P_app_MW,deltaP,changed,I_app_A,V_oc_V,P_reconstructed_MW,soc_next,ok_power_bounds,ok_soc_bounds,ok_power_consistency,blocked_charge_at_max,blocked_discharge_at_min
0,0.200,-0.186350,-0.186350,8.326673e-17,False,-135.570757,1352.000000,-0.186350,0.439670,True,True,True,False,False
1,0.200,-0.186350,-0.186350,8.326673e-17,False,-135.570757,1352.000000,-0.186350,0.439670,True,True,True,False,False
2,0.200,-0.093175,-0.093175,4.163336e-17,False,-68.341581,1352.000000,-0.093175,0.320818,True,True,True,False,False
3,0.200,-0.093175,-0.093175,4.163336e-17,False,-68.341581,1352.000000,-0.093175,0.320818,True,True,True,False,False
4,0.200,0.000000,0.000000,0.000000e+00,False,0.000000,1352.000000,0.000000,0.200000,True,True,True,False,False
5,0.200,0.000000,0.000000,0.000000e+00,False,0.000000,1352.000000,0.000000,0.200000,True,True,True,False,False
6,0.200,0.093175,0.000000,-9.317500e-02,True,0.000000,1352.000000,0.000000,0.200000,True,True,True,False,True
7,0.200,0.093175,0.000000,-9.317500e-02,True,0.000000,1352.000000,0.000000,0.200000,True,True,True,False,True
8,0.200,0.186350,0.000000,-1.863500e-01,True,0.000000,1352.000000,0.000000,0.200000,True,True,True,False,True
9,0.200,0.186350,0.000000,-1.863500e-01,True,0.000000,1352.000000,0.000000,0.200000,True,True,True,False,True


num tests: 35 | num fails: 0


,soc,P_req_MW,P_app_MW,deltaP,changed,I_app_A,V_oc_V,P_reconstructed_MW,soc_next,ok_power_bounds,ok_soc_bounds,ok_power_consistency,blocked_charge_at_max,blocked_discharge_at_min


# 5. Day-Ahead (DA) publication & planning logic

In the overlap (realistic) environment, the key market assumption is:

• The agent dispatches in real time every 30 minutes
• The DA prices for tomorrow are not known all the time
• They only become available after the DA publication time (e.g. 13:00)
• Only after that moment should the agent:
	•	see the tomorrow DA curve in the observation
	•	be able to write meaningful values into tomorrow_plan

In [13]:
import numpy as np
import pandas as pd

def run_da_publication_sanity(env, seed=123, n_steps=80):
    obs, reset_info = env.reset(seed=seed)
    rows = []

    for k in range(n_steps):
        obs_before = obs.copy()  # s_t

        action = env.action_space.sample()
        obs, r, term, trunc, info = env.step(action)  # obs is now s_{t+1}

        # DA curve visibility should be evaluated from s_t (obs_before)
        da_curve = obs_before[6:]
        curve_nonzero = bool(np.any(np.abs(da_curve) > 1e-6))

        rows.append({
            "k": k,
            "trade_ts": info["trade_ts"],        # current step time (trade clock)
            "tau": info["tau"],
            "da_available": info["da_available"],
            "tomorrow_curve_nonzero": curve_nonzero,
        })

        if term or trunc:
            break

    df = pd.DataFrame(rows)
    df["trade_ts"] = pd.to_datetime(df["trade_ts"])
    df["dt_min"] = df["trade_ts"].diff().dt.total_seconds() / 60.0

    # Expected publication time for the starting trade day
    trade_day0 = df["trade_ts"].iloc[0].floor("D")
    df["publish_ts_expected"] = trade_day0 + pd.Timedelta(hours=env.publish_hour)

    # Helpful: where the gate first flips
    df["gate_mismatch"] = df["da_available"] != df["tomorrow_curve_nonzero"]

    return df

df_da = run_da_publication_sanity(env, seed=123, n_steps=80)
df_da

,k,trade_ts,tau,da_available,tomorrow_curve_nonzero,dt_min,publish_ts_expected,gate_mismatch
0,0,2022-02-19 00:00:00,1,False,False,NaN,2022-02-19 13:00:00,False
1,1,2022-02-19 00:30:00,2,False,False,30.0,2022-02-19 13:00:00,False
2,2,2022-02-19 01:00:00,3,False,False,30.0,2022-02-19 13:00:00,False
3,3,2022-02-19 01:30:00,4,False,False,30.0,2022-02-19 13:00:00,False
4,4,2022-02-19 02:00:00,5,False,False,30.0,2022-02-19 13:00:00,False
...,...,...,...,...,...,...,...,...
75,75,2022-02-20 13:30:00,28,True,True,30.0,2022-02-19 13:00:00,False
76,76,2022-02-20 14:00:00,29,True,True,30.0,2022-02-19 13:00:00,False
77,77,2022-02-20 14:30:00,30,True,True,30.0,2022-02-19 13:00:00,False
78,78,2022-02-20 15:00:00,31,True,True,30.0,2022-02-19 13:00:00,False


# 6. Step Level Invariants
## 6.1. SoC always within bounds

In [14]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)
eps = 1e-6

N_steps = 20_000  # set to len(data) if you want a full pass

obs, info = env.reset()
fails = []

for t in range(N_steps):
    a = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(a)

    soc = float(obs[0])
    if not (env.SoC_min - eps <= soc <= env.SoC_max + eps):
        fails.append({
            "t": t,
            "action": int(a),
            "soc": soc,
            "SoC_min": env.SoC_min,
            "SoC_max": env.SoC_max,
            "P_requested_MW": float(env.power_levels[a]),
            "P_applied_MW": float(info.get("P_applied_MW", np.nan)),
            "reward": float(reward),
        })
        break  # stop at first failure for fast debugging

    if terminated or truncated:
        break

print(f"ran steps: {t+1}")
print(f"num fails: {len(fails)}")
if fails:
    display(pd.DataFrame(fails))

ran steps: 1440
num fails: 0


## 6.2. Energy Accounting Check

In [15]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)
eps = 1e-9

obs, info = env.reset()
rows = []
fails = []

N = 30

for t in range(N):
    a = env.action_space.sample()
    P_req = float(env.power_levels[dispatch_idx])
    
    soc_before = env.soc
    obs, reward, terminated, truncated, info = env.step(a)
    soc_after = env.soc

    P_app = float(info["P_applied_MW"])
    E_MWh = P_app * env.dt_hours

    ok_mag = (abs(E_MWh) <= env.P_max_MW * env.dt_hours + 1e-6)

    ok_sign = ((E_MWh >= -eps) if (P_app >= 0) else (E_MWh <= eps)) 

    dispatch_idx = int(a[0])
    plan_idx = int(a[1])
    
    row = {
        "t": t,
        "dispatch_action": dispatch_idx,
        "plan_idx":plan_idx,
        "P_req_MW": P_req,
        "P_app_MW": P_app,
        "soc_before": soc_before,
        "soc_after": soc_after,
        "delta_soc": delta,
        "dt_hours": float(env.dt_hours),
        "E_MWh" : E_MWh,
        "ok mag": ok_mag,
        "ok_sign": ok,
    }

    rows.append(row)
    if not ok:
        fails.append(row)
    if terminated or truncated:
        break

df = pd.DataFrame(rows)
display(df)

    

,t,dispatch_action,plan_idx,P_req_MW,P_app_MW,soc_before,soc_after,delta_soc,dt_hours,E_MWh,ok mag,ok_sign
0,0,3,0,0.18635,-0.074540,0.339794,0.435534,0.0,0.5,-0.037270,True,True
1,1,3,10,-0.07454,-0.074540,0.435534,0.531225,0.0,0.5,-0.037270,True,True
2,2,5,6,-0.07454,0.000000,0.531225,0.531225,0.0,0.5,0.000000,True,True
3,3,2,5,0.00000,-0.111810,0.531225,0.674162,0.0,0.5,-0.055905,True,True
4,4,3,0,-0.11181,-0.074540,0.674162,0.768738,0.0,0.5,-0.037270,True,True
5,5,6,1,-0.07454,0.037270,0.768738,0.720529,0.0,0.5,0.018635,True,True
6,6,10,2,0.03727,0.186350,0.720529,0.476224,0.0,0.5,0.093175,True,True
7,7,1,3,0.18635,-0.149080,0.476224,0.666331,0.0,0.5,-0.074540,True,True
8,8,2,10,-0.14908,-0.111810,0.666331,0.807760,0.0,0.5,-0.055905,True,True
9,9,3,0,-0.11181,-0.072731,0.807760,0.900000,0.0,0.5,-0.036366,True,True


# 7. Reward Decomposition Checks

## 6.1. Profit term sanity with constant price

In [16]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)
eps = 1e-6

# Make carbon term vanish
env.lambda_ci = 0.0

obs, info = env.reset(seed=123)

rows = []
fails = []

N = 50

for t in range(N):
    a = env.action_space.sample()
    dispatch_idx = int(a[0])
    plan_idx = int(a[1])

    soc_before = float(env.soc)
    obs, reward, terminated, truncated, info = env.step(a)
    soc_after = float(env.soc)

    P_app = float(info["P_applied_MW"])
    price_now = float(info["price_now"])
    dt = float(env.dt_hours)

    # Profit definition used in env: profit = E_MWh * price_now
    E_MWh = P_app * dt
    profit_calc = E_MWh * price_now

    ok_profit_only = np.isclose(float(reward), profit_calc, atol=1e-5)

    row = {
        "t": t,
        "dispatch_idx": dispatch_idx,
        "plan_idx": plan_idx,
        "P_app_MW": P_app,
        "price_now": price_now,
        "E_MWh": E_MWh,
        "reward_env": float(reward),
        "profit_calc": profit_calc,
        "ok_profit_only": ok_profit_only,
        "soc_before": soc_before,
        "soc_after": soc_after,
        "tau": info.get("tau", None),
        "trade_ts": info.get("trade_ts", None),
    }
    rows.append(row)

    if not ok_profit_only:
        fails.append(row)
        break

    if terminated or truncated:
        break

df = pd.DataFrame(rows)
display(df)

print("num fails:", len(fails))
if fails:
    display(pd.DataFrame(fails))

,t,dispatch_idx,plan_idx,P_app_MW,price_now,E_MWh,reward_env,profit_calc,ok_profit_only,soc_before,soc_after,tau,trade_ts
0,0,1,10,-1.490800e-01,100.000000,-7.454000e-02,-7.454000e+00,-7.454000e+00,True,0.572941,0.762595,1,2022-02-19 00:00:00
1,1,8,1,1.118100e-01,112.000000,5.590500e-02,6.261360e+00,6.261360e+00,True,0.762595,0.617006,2,2022-02-19 00:30:00
2,2,3,8,-7.454000e-02,140.000000,-3.727000e-02,-5.217800e+00,-5.217800e+00,True,0.617006,0.712056,3,2022-02-19 01:00:00
3,3,7,3,7.454000e-02,80.000000,3.727000e-02,2.981600e+00,2.981600e+00,True,0.712056,0.615310,4,2022-02-19 01:30:00
4,4,4,1,-3.727000e-02,97.599998,-1.863500e-02,-1.818776e+00,-1.818776e+00,True,0.615310,0.663000,5,2022-02-19 02:00:00
5,5,0,4,-1.863500e-01,70.000000,-9.317500e-02,-6.522250e+00,-6.522250e+00,True,0.663000,0.897261,6,2022-02-19 02:30:00
6,6,3,6,-2.147562e-03,88.400002,-1.073781e-03,-9.492224e-02,-9.492224e-02,True,0.897261,0.900000,7,2022-02-19 03:00:00
7,7,8,0,1.118100e-01,53.000000,5.590500e-02,2.962965e+00,2.962965e+00,True,0.900000,0.754489,8,2022-02-19 03:30:00
8,8,3,3,-7.454000e-02,70.000000,-3.727000e-02,-2.608900e+00,-2.608900e+00,True,0.754489,0.849034,9,2022-02-19 04:00:00
9,9,2,8,-4.007517e-02,50.000000,-2.003759e-02,-1.001879e+00,-1.001879e+00,True,0.849034,0.900000,10,2022-02-19 04:30:00


num fails: 0


## 7.2. Reward Check including Carbon 

In [17]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)
eps = 1e-6

env.lambda_ci = float(env.lambda_ci)  # keep as configured

obs, info = env.reset(seed=123)

rows = []
fails = []

N = 50

for t in range(N):
    a = env.action_space.sample()
    dispatch_idx = int(a[0])
    plan_idx = int(a[1])

    obs, reward, terminated, truncated, info = env.step(a)

    P_app = float(info["P_applied_MW"])
    price_now = float(info["price_now"])
    ci_now = float(info["ci_now"])
    dt = float(env.dt_hours)

    E_MWh = P_app * dt
    profit_calc = E_MWh * price_now

    E_imp_kWh = max(-E_MWh, 0.0) * 1000.0
    E_exp_kWh = max(E_MWh, 0.0) * 1000.0
    carbon_penalty_calc = env.lambda_ci * (E_imp_kWh - E_exp_kWh) * ci_now

    reward_calc = profit_calc - carbon_penalty_calc

    ok_reward = np.isclose(float(reward), reward_calc, atol=1e-5)

    row = {
        "t": t,
        "dispatch_idx": dispatch_idx,
        "plan_idx": plan_idx,
        "P_app_MW": P_app,
        "price_now": price_now,
        "ci_now": ci_now,
        "E_MWh": E_MWh,
        "profit_calc": profit_calc,
        "carbon_penalty_calc": carbon_penalty_calc,
        "reward_env": float(reward),
        "reward_calc": reward_calc,
        "ok_reward": ok_reward,
        "tau": info.get("tau", None),
        "trade_ts": info.get("trade_ts", None),
    }
    rows.append(row)

    if not ok_reward:
        fails.append(row)
        break

    if terminated or truncated:
        break

df = pd.DataFrame(rows)
display(df)

print("num fails:", len(fails))
if fails:
    display(pd.DataFrame(fails))

,t,dispatch_idx,plan_idx,P_app_MW,price_now,ci_now,E_MWh,profit_calc,carbon_penalty_calc,reward_env,reward_calc,ok_reward,tau,trade_ts
0,0,9,1,1.490800e-01,100.000000,48.0,7.454000e-02,7.454000e+00,-3.220128e+03,3.227582e+03,3.227582e+03,True,1,2022-02-19 00:00:00
1,1,2,6,-1.118100e-01,112.000000,49.0,-5.590500e-02,-6.261360e+00,2.465410e+03,-2.471672e+03,-2.471672e+03,True,2,2022-02-19 00:30:00
2,2,0,9,-1.863500e-01,140.000000,54.0,-9.317500e-02,-1.304450e+01,4.528305e+03,-4.541349e+03,-4.541349e+03,True,3,2022-02-19 01:00:00
3,3,5,0,0.000000e+00,80.000000,53.0,0.000000e+00,0.000000e+00,-0.000000e+00,0.000000e+00,0.000000e+00,True,4,2022-02-19 01:30:00
4,4,7,2,7.454000e-02,97.599998,54.0,3.727000e-02,3.637552e+00,-1.811322e+03,1.814959e+03,1.814959e+03,True,5,2022-02-19 02:00:00
5,5,6,9,3.727000e-02,70.000000,56.0,1.863500e-02,1.304450e+00,-9.392040e+02,9.405084e+02,9.405084e+02,True,6,2022-02-19 02:30:00
6,6,9,7,1.490800e-01,88.400002,57.0,7.454000e-02,6.589336e+00,-3.823902e+03,3.830491e+03,3.830491e+03,True,7,2022-02-19 03:00:00
7,7,4,6,-3.727000e-02,53.000000,57.0,-1.863500e-02,-9.876550e-01,9.559755e+02,-9.569631e+02,-9.569631e+02,True,8,2022-02-19 03:30:00
8,8,5,0,0.000000e+00,70.000000,58.0,0.000000e+00,0.000000e+00,-0.000000e+00,0.000000e+00,0.000000e+00,True,9,2022-02-19 04:00:00
9,9,2,3,-1.118100e-01,50.000000,57.0,-5.590500e-02,-2.795250e+00,2.867926e+03,-2.870722e+03,-2.870722e+03,True,10,2022-02-19 04:30:00


num fails: 0


# 8. Plan Rollover
## 8.1. Yesterday plan becomes today_plan at rollover

In [18]:
env = BatteryEnv(env_config)
obs, info = env.reset(seed=123)

pattern = np.arange(48) % env.n_power_levels
env.tomorrow_plan[:] = pattern.copy()

# Freeze DA availability (so step() cannot overwrite tomorrow_plan)
env.da_publish_ts[:] = env.trade_ts + np.timedelta64(3650, "D")  # +10 years

# Fast-forward to rollover
steps_left = 48 - env.slot0
for _ in range(steps_left):
    obs, r, term, trunc, inf = env.step(env.action_space.sample())

ok = np.all(env.today_plan == pattern)
print("today_plan matches yesterday's tomorrow_plan:", ok)
if not ok:
    print("max abs diff:", np.max(np.abs(env.today_plan - pattern)))

today_plan matches yesterday's tomorrow_plan: True


In [19]:
env = BatteryEnv(env_config, publish_hour=0)  # DA available from 00:00
obs, info = env.reset(seed=123)

pattern = np.arange(48) % env.n_power_levels

for _ in range(48):
    tau0 = env.slot0              # 0..47
    plan_idx = int(pattern[tau0])
    dispatch_idx = 0
    action = np.array([dispatch_idx, plan_idx], dtype=int)
    obs, r, term, trunc, inf = env.step(action)

ok = np.all(env.today_plan == pattern)
print("today_plan matches yesterday's tomorrow_plan:", ok)
if not ok:
    print("max abs diff:", np.max(np.abs(env.today_plan - pattern)))
    print("today_plan head:", env.today_plan[:10])
    print("pattern head:", pattern[:10])

today_plan matches yesterday's tomorrow_plan: True


In [20]:
obs, info = env.reset(seed=123)
print("First step da_available:", env._da_available_now(int(env.current_day_idxs[env.slot0])))
print("First trade_ts:", env.trade_ts[int(env.current_day_idxs[env.slot0])])
print("First da_publish_ts:", env.da_publish_ts[int(env.current_day_idxs[env.slot0])])

First step da_available: True
First trade_ts: 2022-02-19T00:00:00.000000000
First da_publish_ts: 2022-02-19T00:00:00.000000000


In [21]:
obs, info = env.reset(seed=123)
flags = []
for _ in range(48):
    idx = int(env.current_day_idxs[env.slot0])
    flags.append(env._da_available_now(idx))
    obs, r, term, trunc, inf = env.step(env.action_space.sample())
print("Any DA available today?", any(flags))
print("Count true:", sum(flags))

Any DA available today? True
Count true: 48


In [22]:
env2 = BatteryEnv(env_config, publish_hour=13)
obs, info = env2.reset(seed=123)

idx0 = int(env2.current_day_idxs[env2.slot0])
print("trade_ts:", env2.trade_ts[idx0])
print("da_publish_ts:", env2.da_publish_ts[idx0])
print("da_available:", env2._da_available_now(idx0))

trade_ts: 2022-02-19T00:00:00.000000000
da_publish_ts: 2022-02-19T13:00:00.000000000
da_available: False


In [23]:
import numpy as np
import pandas as pd

def action_for_power_idx(env, target_MW: float) -> int:
    return int(np.argmin(np.abs(env.power_levels - target_MW)))

def run_designB_tests(env, seed=123, publish_hour=13):
    # make sure we're using intended publish hour
    env.publish_hour = publish_hour

    obs, info = env.reset(seed=seed)
    nA = env.n_power_levels

    # --- helpers to locate "now" ---
    def cur_idx():
        return int(env.current_day_idxs[env.slot0])

    def cur_tau0(idx):
        return int(env.tau[idx]) - 1  # 0..47

    # ----------------------------
    # TEST 1: "tomorrow_plan" is gated by da_available
    # ----------------------------
    # We'll try to write a known pattern after publish time; before publish it must NOT change.
    idle = action_for_power_idx(env, 0.0)
    pattern = np.arange(48) % nA

    # Track changes to tomorrow_plan over one trade day
    before_changes = 0
    after_changes = 0

    # Walk one full trade day (48 steps)
    for _ in range(48):
        idx = cur_idx()
        tau0 = cur_tau0(idx)

        da_avail = env._da_available_now(idx)

        old = int(env.tomorrow_plan[tau0])
        # dispatch idle, plan attempts to write pattern[tau0]
        a = np.array([idle, int(pattern[tau0])], dtype=int)
        obs, r, term, trunc, inf = env.step(a)

        new = int(env.tomorrow_plan[tau0])

        if da_avail:
            after_changes += (new != old)
        else:
            before_changes += (new != old)

        if term or trunc:
            break

    print("TEST 1 (DA gate)")
    print("  any writes before publish? ", before_changes > 0)
    print("  any writes after publish?  ", after_changes > 0)

    # ----------------------------
    # TEST 2: rollover: today_plan == yesterday's tomorrow_plan
    # ----------------------------
    # At this point we've advanced one full day already, so the rollover occurred.
    # The env should have shifted tomorrow_plan -> today_plan at day boundary.
    # But tomorrow_plan has been reset to 0 after rollover, so we compare today_plan to the pattern we attempted to write.

    today_ok = np.all(env.today_plan == pattern)
    max_diff = np.max(np.abs(env.today_plan - pattern))
    print("\nTEST 2 (rollover plan)")
    print("  today_plan matches pattern:", today_ok)
    print("  max abs diff:", int(max_diff))
    print("  today_plan head:", env.today_plan[:10])
    print("  pattern head:   ", pattern[:10])

    # ----------------------------
    # TEST 3: execution day dispatch uses today_plan (Design B core)
    # ----------------------------
    # Under Design B, on the *delivery day*, dispatch power should come from today_plan[tau]
    # regardless of dispatch_idx chosen by the agent.
    #
    # We'll take random dispatch_idx but compute expected power_idx = today_plan[tau0]
    # and check that info["P_applied_MW"] matches that expected level.

    rows = []
    fails = 0

    for _ in range(48):
        idx = cur_idx()
        tau0 = cur_tau0(idx)

        # random dispatch_idx (should be ignored under Design B)
        dispatch_idx = env.action_space.sample()[0]
        plan_idx = 0  # irrelevant on execution day, but keep valid

        a = np.array([int(dispatch_idx), int(plan_idx)], dtype=int)
        soc_before = float(env.soc)

        obs, r, term, trunc, inf = env.step(a)
        soc_after = float(env.soc)

        expected_power_idx = int(env.today_plan[tau0])
        expected_P = float(env.power_levels[expected_power_idx])
        got_P = float(inf["P_applied_MW"])

        ok = np.isclose(got_P, expected_P, atol=1e-6)

        rows.append({
            "trade_ts": inf.get("trade_ts", None),
            "tau": int(tau0+1),
            "dispatch_idx_agent": int(dispatch_idx),
            "expected_power_idx_from_today_plan": expected_power_idx,
            "expected_P_MW": expected_P,
            "got_P_MW": got_P,
            "ok_exec": ok,
            "soc_before": soc_before,
            "soc_after": soc_after,
        })

        fails += (not ok)

        if term or trunc:
            break

    df = pd.DataFrame(rows)

    print("\nTEST 3 (execution uses today_plan)")
    print("  fails:", int(fails), "out of", len(df))

    return df

# usage:
df_exec = run_designB_tests(env, seed=123, publish_hour=13)
display(df_exec.head(20))
display(df_exec[df_exec["ok_exec"] == False].head(20))

TEST 1 (DA gate)
  any writes before publish?  False
  any writes after publish?   True

TEST 2 (rollover plan)
  today_plan matches pattern: True
  max abs diff: 0
  today_plan head: [0 1 2 3 4 5 6 7 8 9]
  pattern head:    [0 1 2 3 4 5 6 7 8 9]

TEST 3 (execution uses today_plan)
  fails: 41 out of 48


,trade_ts,tau,dispatch_idx_agent,expected_power_idx_from_today_plan,expected_P_MW,got_P_MW,ok_exec,soc_before,soc_after
0,2022-02-20 00:00:00,1,1,0,-0.18635,-1.490800e-01,False,0.572941,0.762595
1,2022-02-20 00:30:00,2,1,1,-0.14908,-1.086509e-01,False,0.762595,0.900000
2,2022-02-20 01:00:00,3,10,2,-0.11181,1.863500e-01,False,0.900000,0.655846
3,2022-02-20 01:30:00,4,8,3,-0.07454,1.118100e-01,False,0.655846,0.510168
4,2022-02-20 02:00:00,5,5,4,-0.03727,0.000000e+00,False,0.510168,0.510168
5,2022-02-20 02:30:00,6,9,5,0.00000,1.490800e-01,False,0.510168,0.313049
6,2022-02-20 03:00:00,7,9,6,0.03727,8.582789e-02,False,0.313049,0.200000
7,2022-02-20 03:30:00,8,9,7,0.07454,0.000000e+00,False,0.200000,0.200000
8,2022-02-20 04:00:00,9,10,8,0.11181,0.000000e+00,False,0.200000,0.200000
9,2022-02-20 04:30:00,10,7,9,0.14908,0.000000e+00,False,0.200000,0.200000


,trade_ts,tau,dispatch_idx_agent,expected_power_idx_from_today_plan,expected_P_MW,got_P_MW,ok_exec,soc_before,soc_after
0,2022-02-20 00:00:00,1,1,0,-0.18635,-1.490800e-01,False,0.572941,0.762595
1,2022-02-20 00:30:00,2,1,1,-0.14908,-1.086509e-01,False,0.762595,0.900000
2,2022-02-20 01:00:00,3,10,2,-0.11181,1.863500e-01,False,0.900000,0.655846
3,2022-02-20 01:30:00,4,8,3,-0.07454,1.118100e-01,False,0.655846,0.510168
4,2022-02-20 02:00:00,5,5,4,-0.03727,0.000000e+00,False,0.510168,0.510168
5,2022-02-20 02:30:00,6,9,5,0.00000,1.490800e-01,False,0.510168,0.313049
6,2022-02-20 03:00:00,7,9,6,0.03727,8.582789e-02,False,0.313049,0.200000
7,2022-02-20 03:30:00,8,9,7,0.07454,0.000000e+00,False,0.200000,0.200000
8,2022-02-20 04:00:00,9,10,8,0.11181,0.000000e+00,False,0.200000,0.200000
9,2022-02-20 04:30:00,10,7,9,0.14908,0.000000e+00,False,0.200000,0.200000
